In [1]:
## load libraries
import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader

load_dotenv()

True

In [2]:
# Install the pinned dependencies used by this notebook.
%pip install --upgrade "langchain==0.3.25" "langchain-core==0.3.59" "langchain-community==0.3.24" "langchain-openai==0.3.17" "langchain-text-splitters==0.3.8" "langchain-groq==0.2.5" "openai==1.68.2" "tiktoken>=0.13.0" "python-dotenv>=1.2.2" "numpy>=2.0.0"

  Using cached langchain_core-0.3.59-py3-none-any.whl.metadata (5.9 kB)
  Using cached langchain_openai-0.3.17-py3-none-any.whl.metadata (2.3 kB)
  Using cached langchain_groq-0.2.5-py3-none-any.whl.metadata (2.6 kB)
Using cached langchain_core-0.3.59-py3-none-any.whl (437 kB)
Using cached langchain_openai-0.3.17-py3-none-any.whl (62 kB)
Using cached langchain_groq-0.2.5-py3-none-any.whl (15 kB)

  Attempting uninstall: langchain-core

    Found existing installation: langchain-core 0.3.59

    Uninstalling langchain-core-0.3.59:

   ---------------------------------------- 0/3 [langchain-core]

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\users\\raavi\\appdata\\local\\programs\\python\\python312\\lib\\site-packages\\langchain_core\\callbacks\\base.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
!python -m pip install --upgrade --force-reinstall langchain==0.3.25 langchain-core==0.3.59 langchain-openai==0.3.17 langchain-community==0.3.24 langchain-text-splitters==0.3.8 openai==1.82.0

  Using cached langchain_core-0.3.59-py3-none-any.whl.metadata (5.9 kB)
  Using cached langchain_openai-0.3.17-py3-none-any.whl.metadata (2.3 kB)
  Using cached openai-1.82.0-py3-none-any.whl.metadata (25 kB)
  Using cached langsmith-0.3.45-py3-none-any.whl.metadata (15 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached numpy-2.4.6-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached zstandard-0.23.0-cp312-cp312-win_amd64.whl.metadata (3.0 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 24.2 MB/s  0:00:00
Using cached langchain_core-0.3.59-py3-none-any.whl (437 kB)
Using cached langchain_openai-0.3.17-py3-none-any.whl (62 kB)
Using cached openai-1.82.0-py3-none-any.whl (720 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 48.2 MB/s  0:00:00
Using cached langsmith-0.3.45-py3-none-any.whl (363

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.7 requires langchain-core<2.0.0,>=1.3.3, but you have langchain-core 0.3.59 which is incompatible.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.8 which is incompatible.
langchain-huggingface 1.2.2 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.3.59 which is incompatible.
langgraph 1.2.4 requires langch

In [12]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

### Data ingestion and parsing

In [4]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

In [5]:
### Text splitting
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100, 
    chunk_overlap=20, 
    length_function=len, 
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(sample_documents)
print(f"Number of chunks created: {len(chunks)}")
print(chunks[0])
print(chunks[1])
print(chunks[2])


Number of chunks created: 13
page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
page_content='These systems are designed to think like humans and mimic their actions.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
page_content='AI can be categorized into narrow AI and general AI.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [6]:
print(f"Created {len(chunks)} chunks from {len(sample_documents)} documents")
print("\nExample chunk:")
print(f"Content: {chunks[0].page_content}")
print(f"Metadata: {chunks[0].metadata}")

Created 13 chunks from 4 documents

Example chunk:
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
Metadata: {'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [7]:
### loading the embedding model
import os
from dotenv import load_dotenv

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [13]:
# Initialize OpenAI embeddings with the latest model

embeddings=OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1536
)

## Example: create a embedding for a single text
sample_text="What is machine learning"
sample_embedding=embeddings.embed_query(sample_text)
sample_embedding

[-0.005901336669921875,
 -0.005916595458984375,
 0.0005407333374023438,
 -0.0335693359375,
 0.0212249755859375,
 0.0221099853515625,
 -0.0008015632629394531,
 0.00928497314453125,
 -0.0226287841796875,
 0.03790283203125,
 0.01519012451171875,
 -0.03668212890625,
 -0.034027099609375,
 0.0015821456909179688,
 0.0267486572265625,
 0.015960693359375,
 0.0008187294006347656,
 -0.0308990478515625,
 0.02337646484375,
 0.00489044189453125,
 8.058547973632812e-05,
 0.032470703125,
 0.042236328125,
 -0.0293121337890625,
 0.0159759521484375,
 -0.0209503173828125,
 0.0266265869140625,
 0.01239776611328125,
 0.0074005126953125,
 -0.006023406982421875,
 -0.00939178466796875,
 -0.029052734375,
 -0.031341552734375,
 0.027557373046875,
 0.041748046875,
 0.0004277229309082031,
 -0.00807952880859375,
 -0.0106658935546875,
 -0.0555419921875,
 0.0026226043701171875,
 -0.056304931640625,
 -0.01629638671875,
 0.03021240234375,
 0.07611083984375,
 -0.0260009765625,
 -0.048828125,
 -0.036102294921875,
 -0.0353

In [15]:
texts = ["AI", "Machine Learning", "Deep Learning", "Natural Language Processing"]
batch_embeddings = embeddings.embed_documents(texts)
print(f"Batch embeddings shape: {np.array(batch_embeddings).shape}")
print(batch_embeddings[0])

Batch embeddings shape: (4, 1536)
[-0.0081634521484375, -0.0246124267578125, 0.0029850006103515625, 0.0251617431640625, 0.006565093994140625, -0.028228759765625, -0.005023956298828125, 0.020904541015625, -0.036895751953125, 0.01279449462890625, -0.0030364990234375, -0.020111083984375, 0.0002522468566894531, -0.03271484375, 0.0064544677734375, -0.0252685546875, -0.031097412109375, -0.054412841796875, 0.03277587890625, -0.0184173583984375, 0.01666259765625, 0.04833984375, -0.024871826171875, 0.01438140869140625, 0.0293426513671875, 0.004047393798828125, 0.00928497314453125, 0.01337432861328125, 0.0025310516357421875, -0.0225372314453125, 0.0321044921875, -0.0280303955078125, 0.005359649658203125, -0.038177490234375, -0.0167236328125, 0.01434326171875, -0.038604736328125, -0.01038360595703125, -0.0105438232421875, -0.0191650390625, 0.0321044921875, 0.014556884765625, -0.021514892578125, 0.0160675048828125, -0.01186370849609375, 0.0013990402221679688, -0.004833221435546875, -0.033660888671

In [17]:
def compare_embeddings(embedding1, embedding2):
    # Calculate cosine similarity
    dot_product = np.dot(embedding1, embedding2)
    norm1 = np.linalg.norm(embedding1)
    norm2 = np.linalg.norm(embedding2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)

print("Cosine similarity between 'AI' and 'Machine Learning':", compare_embeddings(batch_embeddings[0], batch_embeddings[1]))
print("Cosine similarity between 'AI' and 'Deep Learning':", compare_embeddings(batch_embeddings[0], batch_embeddings[2]))
print("Cosine similarity between 'AI' and 'Natural Language Processing':", compare_embeddings(batch_embeddings[0], batch_embeddings[3]))
print("Cosine similarity between 'Machine Learning' and 'Deep Learning':", compare_embeddings(batch_embeddings[1], batch_embeddings[2]))
print("Cosine similarity between 'Machine Learning' and 'Natural Language Processing':", compare_embeddings(batch_embeddings[1], batch_embeddings[3]))
print("Cosine similarity between 'Deep Learning' and 'Natural Language Processing':", compare_embeddings(batch_embeddings[2], batch_embeddings[3]))
print("Cosine similarity between 'AI' and itself:", compare_embeddings(batch_embeddings[0], batch_embeddings[0]))

Cosine similarity between 'AI' and 'Machine Learning': 0.3723865165883862
Cosine similarity between 'AI' and 'Deep Learning': 0.3163242144059092
Cosine similarity between 'AI' and 'Natural Language Processing': 0.2671217637449197
Cosine similarity between 'Machine Learning' and 'Deep Learning': 0.6968920892049772
Cosine similarity between 'Machine Learning' and 'Natural Language Processing': 0.4446134214572798
Cosine similarity between 'Deep Learning' and 'Natural Language Processing': 0.45047900602743307
Cosine similarity between 'AI' and itself: 1.0


In [18]:
print("Cosine similarity between 'AI' and 'pizza':", compare_embeddings(batch_embeddings[0], batch_embeddings[1]))

Cosine similarity between 'AI' and 'pizza': 0.3723865165883862


In [20]:
print(f"AI vs Machine Learning similarity: {compare_embeddings(batch_embeddings[0], batch_embeddings[3])}")

AI vs Machine Learning similarity: 0.2671217637449197


### Create Faiss Vector Store

In [22]:
!pip install faiss-cpu

   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ---------- ----------------------------- 4.2/16.1 MB 41.8 MB/s eta 0:00:01
   ---------------------------------------  15.7/16.1 MB 52.1 MB/s eta 0:00:01
   ---------------------------------------- 16.1/16.1 MB 48.3 MB/s  0:00:00


In [26]:
vector_store = FAISS.from_documents(chunks, embeddings)
print(f"Number of documents in vector store: {vector_store.index.ntotal} vectors")

Number of documents in vector store: 13 vectors


In [27]:
vector_store

In [28]:
## Save and load vector store

vector_store.save_local("faiss_index")

In [29]:
## load the vector store
loaded_vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
print(f"Number of documents in loaded vector store: {loaded_vector_store.index.ntotal} vectors")

Number of documents in loaded vector store: 13 vectors


In [34]:
## similarity search

query = "What is deep learning?"

results = vector_store.similarity_search(query, k=2)
print(results)


[Document(id='a4950e38-b953-41ba-95fa-7baf84761a72', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.'), Document(id='a5491273-74db-451e-ae47-bd3cb7a6c007', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep learning has revolutionized computer vision, NLP, and speech recognition.')]


In [35]:
print(f"query: {query}\n")
print("Top 2 similar documents:")

for i, result in enumerate(results):
    print(f"\nResult {i+1}:")
    print(f"Content: {result.page_content}")
    print(f"Metadata: {result.metadata}")   

query: What is deep learning?

Top 2 similar documents:

Result 1:
Content: Deep Learning is a subset of machine learning based on artificial neural networks.
Metadata: {'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}

Result 2:
Content: Deep learning has revolutionized computer vision, NLP, and speech recognition.
Metadata: {'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}


In [36]:
### similarity search with score
print("\n\nSimilarity search with scores:")
for doc, score in vector_store.similarity_search_with_score(query, k=2):
    print(f"\nContent: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print(f"Similarity Score: {score:.4f}")



Similarity search with scores:

Content: Deep Learning is a subset of machine learning based on artificial neural networks.
Metadata: {'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}
Similarity Score: 0.7492

Content: Deep learning has revolutionized computer vision, NLP, and speech recognition.
Metadata: {'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}
Similarity Score: 0.8114


In [38]:
### search with metadata filter
filtered_results = {"topic": "DL"}
filtered_docs = vector_store.similarity_search(query, k=2, filter=filtered_results)
print(filtered_docs)

[Document(id='a4950e38-b953-41ba-95fa-7baf84761a72', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.'), Document(id='a5491273-74db-451e-ae47-bd3cb7a6c007', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep learning has revolutionized computer vision, NLP, and speech recognition.')]


In [39]:
len(filtered_docs)

2

### Build RAG Chain with lcel

In [42]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("GROQ_API_KEY"))

[API_KEY_REMOVED]


In [45]:
!pip install -U langchain-groq

   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 548.1/548.1 kB 17.9 MB/s  0:00:00

  Attempting uninstall: langchain-core

    Found existing installation: langchain-core 0.3.59

    Uninstalling langchain-core-0.3.59:

      Successfully uninstalled langchain-core-0.3.59

   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.25 requires langchain-core<1.0.0,>=0.3.58, but you have langchain-core 1.4.0 which is incompatible.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.8 which is incompatible.
langchain-community 0.3.24 requires langchain-core<1.0.0,>=0.3.59, but you have langchain-core 1.4.0 which is incompatible.
langchain-openai 0.3.17 requires langchain-core<1.0.0,>=0.3.59, but you have langchain-core 1.4.0 which is incompatible.
langchain-text-splitters 0.3.8 requires langchain-core<1.0.0,>=0.3.51, but you have langchain-core 1.4.0 which is incompatible.
langgraph-sdk 0.4.2 requires websockets<16,>=14, but you have websockets 16.0 which is incompatible.


In [99]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

In [100]:
# 1. Simple RAG Chain with LCEL
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [101]:
## Basic retriever
retriever=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001AE5132FD40>, search_kwargs={'k': 3})

In [102]:
from typing import List
# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [103]:
simple_rag_chain=(
    {"context":retriever | format_docs,"question":RunnablePassthrough() }
    | simple_prompt
    | llm
    |StrOutputParser()

)

In [104]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001AE5132FD40>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x

In [105]:
### Conversational RAg Chain

conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided context to answer questions."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\n\nQuestion: {input}"),
])

In [106]:
def create_conversational_rag():
    """Create a conversational RAG chain with memory"""
    return (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()

In [107]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag=

### streaming RAG chain

In [108]:
### streaming RAG chain
streaming_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
)

print("Modern RAG chains created successfully!")
print("Available chains:")
print("- simple_rag_chain: Basic Q&A")
print("- conversational_rag: Maintains conversation history")
print("- streaming_rag_chain: Supports token streaming")

Modern RAG chains created successfully!
Available chains:
- simple_rag_chain: Basic Q&A
- conversational_rag: Maintains conversation history
- streaming_rag_chain: Supports token streaming


In [109]:
# Test function for different chain types
def test_rag_chains(question: str):
    """Test all RAG chain variants"""
    print(f"Question: {question}")
    print("=" * 80)
    
    # 1. Simple RAG
    print("\n1. Simple RAG Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")



In [111]:
test_rag_chains("What is deep learning vs AI?")

Question: What is deep learning vs AI?

1. Simple RAG Chain:
Answer: Deep Learning is a subset of Machine Learning, and Machine Learning is a subset of AI. Therefore, Deep Learning is a subset of AI, but not the other way around, as AI is a broader field that encompasses more than just Deep Learning.


In [112]:
def test_rag_chains(question: str):
    """Test all RAG chain variants"""
    print(f"Question: {question}")
    print("=" * 80)
    
    # 1. Simple RAG
    print("\n1. Simple RAG Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    print("\n2. Streaming RAG:")
    print("Answer: ", end="", flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    print()

In [113]:
test_rag_chains("What is deep learning vs AI?")

Question: What is deep learning vs AI?

1. Simple RAG Chain:
Answer: Based on the provided context, Deep Learning is a subset of Machine Learning, and Machine Learning is a subset of AI. Therefore, Deep Learning is a subset of AI, but not the other way around. AI is a broader category that encompasses Deep Learning, as well as other areas such as narrow AI and general AI.

2. Streaming RAG:
Answer: Deep Learning is a subset of Machine Learning, and Machine Learning is a subset of AI. Therefore, Deep Learning is a subset of AI, but not the other way around, as AI is a broader field that encompasses more than just Deep Learning.


In [114]:
# Test with multiple questions
test_questions = [
    "What is the difference between AI and Machine Learning?",
    "Explain deep learning in simple terms",
    "How does NLP work?"
]

for question in test_questions:
    print("\n" + "=" * 80 + "\n")
    test_rag_chains(question)



Question: What is the difference between AI and Machine Learning?

1. Simple RAG Chain:
Answer: Based on the provided context, Artificial Intelligence (AI) is the simulation of human intelligence in machines, whereas Machine Learning is a subset of AI that enables systems to learn from data. This implies that AI is a broader concept that encompasses Machine Learning, which is a specific technique used to achieve AI's goals.

2. Streaming RAG:
Answer: Based on the provided context, the difference between AI and Machine Learning is that Artificial Intelligence (AI) is the simulation of human intelligence in machines, whereas Machine Learning is a subset of AI that enables systems to learn from data. In other words, AI is a broader concept that encompasses Machine Learning, which is a specific technique used to achieve AI.


Question: Explain deep learning in simple terms

1. Simple RAG Chain:
Answer: Deep learning is a part of machine learning that uses artificial neural networks to he

In [115]:
## Conversational example
print("\n3. Conversational RAG Example:")
chat_history = []

# First question
q1 = "What is machine learning?"
a1 = conversational_rag.invoke({
    "input": q1,
    "chat_history": chat_history
})

print(f"Q1: {q1}")
print(f"A1: {a1}")


3. Conversational RAG Example:
Q1: What is machine learning?
A1: According to the provided context, Machine Learning is a subset of AI that enables systems to learn from data, and it allows systems to find patterns in data without being explicitly programmed.


In [116]:
# Update history
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])

In [117]:
# Follow-up question
q2 = "How is it different from traditional programming?"
a2 = conversational_rag.invoke({
    "input": q2,
    "chat_history": chat_history
})
print(f"\nQ2: {q2}")
print(f"A2: {a2}")


Q2: How is it different from traditional programming?
A2: According to Document 1 (Source: ML Basics), Machine Learning (ML) is different from traditional programming in that instead of being explicitly programmed, ML algorithms find patterns in data. This means that ML doesn't require manual coding for every possible scenario, but rather learns from the data it's given.
